# EdgentRAG — embedding and generation in one Colab runtime

Use this notebook instead of running the two service notebooks separately. It starts two background API processes and two temporary HTTPS tunnels in **one notebook/runtime**:

| Service | Local port | Colab Secret |
| --- | --- | --- |
| Embedding (`/embed`) | 8001 | `EDGENTRAG_EMBEDDING_API_TOKEN` |
| Generation (`/generate`) | 8003 | `EDGENTRAG_GENERATION_API_TOKEN` |

Add both secrets using Colab's key icon and enable notebook access. Keep your existing embedding token. Generate a separate generation token locally with `python -c 'import secrets; print(secrets.token_urlsafe(32))'`.

Select a GPU runtime if available. By default embeddings use CPU to leave GPU memory for generation; change the device in cell 2 if desired. The local backend, database, Floci and worker still run on your Mac. Generation is not connected to session search yet.

Run cells in order. Colab and Quick Tunnels are temporary development infrastructure, not production hosting. Restarting tunnels changes their URLs. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html).


## 1. Load current code and install both dependency extras
Upload the current project to `/content/ai-eng`, or enter a public HTTPS Git URL. Do not paste repository credentials. The checkout must contain both model services.


In [ ]:
from pathlib import Path
import json
import os
import platform
import re
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from urllib.parse import urlsplit

if sys.version_info < (3, 12):
    raise RuntimeError("This project requires Python 3.12+.")

PROJECT_DIR = Path("/content/ai-eng")
if not PROJECT_DIR.exists():
    repo_url = input("Public project HTTPS Git clone URL: ").strip()
    parsed = urlsplit(repo_url)
    if parsed.scheme != "https" or not parsed.netloc or parsed.username or parsed.password:
        raise ValueError("Use an HTTPS URL without credentials, or upload the project.")
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(PROJECT_DIR)], check=True)

for name in ("embedding", "generation"):
    if not (PROJECT_DIR / f"app/backend/src/edgentrag/{name}/app.py").is_file():
        raise RuntimeError(f"Missing {name} service. Upload/update the current project.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e",
     f"{PROJECT_DIR}/app/backend[embedding,generation]"],
    check=True,
)


## 2. Read both secrets and choose devices
Secrets are never printed. Rerunning this cell does not update an already-running server: stop the services and start them again after changing settings.


In [ ]:
from google.colab import userdata

for name in ("EMBEDDING", "GENERATION"):
    key = f"EDGENTRAG_{name}_API_TOKEN"
    try:
        token = userdata.get(key)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        raise RuntimeError(
            f"Add {key} in Colab Secrets and enable notebook access."
        ) from None
    if not token or not token.strip():
        raise RuntimeError(f"{key} must not be empty.")
    os.environ[key] = token
del token

os.environ["EDGENTRAG_EMBEDDING_DEVICE"] = "cpu"
os.environ["EDGENTRAG_GENERATION_DEVICE"] = "auto"
os.environ["EDGENTRAG_EMBEDDING_MODEL_NAME"] = "sentence-transformers/all-MiniLM-L6-v2"
os.environ["EDGENTRAG_GENERATION_MODEL_NAME"] = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print("Both secrets configured; values hidden.")


## 3. Start both APIs in the background
One process per model, with no reload. Health checks do not load weights. The process registry survives cell reruns; duplicate starts are refused. If startup fails, only processes started by this cell are stopped.


In [ ]:
SERVICES = {"embedding": 8001, "generation": 8003}
if "model_processes" not in globals():
    model_processes = {}
if "model_tunnels" not in globals():
    model_tunnels = {}

def stop_process(process):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)

def wait_for_health(url, process, log_path, attempts=30):
    for _ in range(attempts):
        if process.poll() is not None:
            raise RuntimeError(f"Process exited. Inspect {log_path}.")
        try:
            with urllib.request.urlopen(url + "/health", timeout=5) as response:
                return json.load(response)
        except (urllib.error.URLError, TimeoutError):
            time.sleep(1)
    raise RuntimeError(f"Health not reachable. Inspect {log_path}.")

for name, port in SERVICES.items():
    previous = model_processes.get(name)
    if previous is not None and previous.poll() is None:
        raise RuntimeError(f"{name} is running. Use cleanup before restarting.")
    with socket.socket() as probe:
        try:
            probe.bind(("127.0.0.1", port))
        except OSError:
            raise RuntimeError(f"Port {port} is occupied; stop its server first.") from None

started = []
try:
    for name, port in SERVICES.items():
        log_path = Path(f"/tmp/edgentrag-{name}.log")
        service_env = os.environ.copy()
        other = "GENERATION" if name == "embedding" else "EMBEDDING"
        service_env.pop(f"EDGENTRAG_{other}_API_TOKEN", None)
        with log_path.open("w") as log:
            process = subprocess.Popen(
                [sys.executable, "-m", "uvicorn", f"edgentrag.{name}.app:app",
                 "--host", "127.0.0.1", "--port", str(port)],
                cwd=PROJECT_DIR, env=service_env,
                stdout=log, stderr=subprocess.STDOUT,
            )
        model_processes[name] = process
        started.append(process)
        print(name, wait_for_health(f"http://127.0.0.1:{port}", process, log_path))
except Exception:
    for process in reversed(started):
        stop_process(process)
    raise


## 4. Warm up both models locally
First requests download weights and can take several minutes. Warm-up is sequential. A failure leaves the servers running for inspection/retry; check the relevant `/tmp/edgentrag-embedding.log` or `/tmp/edgentrag-generation.log`. For GPU out-of-memory, keep embeddings on CPU, stop/restart the services, and retry.


In [ ]:
def post_model(name, base_url):
    if name == "embedding":
        route = "/embed"
        payload = {"texts": ["A test document chunk from the combined notebook."]}
    else:
        route = "/generate"
        payload = {"prompt": "Explain semantic search in one sentence.", "max_new_tokens": 64}
    request = urllib.request.Request(
        base_url + route,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": "Bearer " + os.environ[f"EDGENTRAG_{name.upper()}_API_TOKEN"],
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        print(name, exc.code, exc.read().decode())
        raise

for name, port in SERVICES.items():
    result = post_model(name, f"http://127.0.0.1:{port}")
    if name == "embedding":
        print(name, result["model"], "dimensions:", result["dimensions"],
              "vectors:", len(result["embeddings"]))
    else:
        print(name, result)


## 5. Start two tunnels from this same notebook
Each API gets a separate URL so existing routes remain unchanged. Install cloudflared once, then launch both tunnels in the background. `/embed` and `/generate` require their respective tokens; health and API documentation are public. A tunnel-start failure stops both tunnels created by this cell, leaving the APIs running.


In [ ]:
if any(p.poll() is None for p in model_tunnels.values()):
    raise RuntimeError("Tunnels are running. Use cleanup before restarting.")
for name in SERVICES:
    if model_processes[name].poll() is not None:
        raise RuntimeError(f"{name} server exited; inspect its log and restart.")

architecture = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine())
if not architecture:
    raise RuntimeError("Unsupported CPU architecture.")
package_path = Path("/tmp/edgentrag-models-cloudflared.deb")
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    f"cloudflared-linux-{architecture}.deb",
    package_path,
)
subprocess.run(["dpkg", "-i", str(package_path)], check=True)

MODEL_URLS = {}
started_tunnels = []
try:
    for name, port in SERVICES.items():
        log_path = Path(f"/tmp/edgentrag-{name}-tunnel.log")
        tunnel_env = os.environ.copy()
        for service in SERVICES:
            tunnel_env.pop(f"EDGENTRAG_{service.upper()}_API_TOKEN", None)
        with log_path.open("w") as log:
            process = subprocess.Popen(
                ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
                env=tunnel_env, stdout=log, stderr=subprocess.STDOUT,
            )
        model_tunnels[name] = process
        started_tunnels.append(process)
        for _ in range(60):
            if process.poll() is not None:
                break
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_path.read_text())
            if match:
                MODEL_URLS[name] = match.group(0)
                break
            time.sleep(1)
        if name not in MODEL_URLS:
            raise RuntimeError(f"No {name} tunnel URL. Inspect {log_path}.")
        print(name, wait_for_health(MODEL_URLS[name], process, log_path))
except Exception:
    for process in reversed(started_tunnels):
        stop_process(process)
    MODEL_URLS.clear()
    raise

print("Local backend .env setting (restart API and worker after updating):")
print("EDGENTRAG_EMBEDDING_SERVICE_URL=" + MODEL_URLS["embedding"])
print("Generation URL for direct testing; backend integration is a later module:")
print("GENERATION_URL=" + MODEL_URLS["generation"])


## 6. Test both URLs from your Mac

Leave this single notebook runtime running. Update `EDGENTRAG_EMBEDDING_SERVICE_URL` in your local backend `.env` to the printed embedding URL, retain the matching `EDGENTRAG_EMBEDDING_API_TOKEN`, and restart the local API and worker. Do not commit secrets. The generation URL is for direct calls for now.

In your local **zsh** terminal:

~~~zsh
EMBEDDING_URL="https://paste-embedding-url.trycloudflare.com"
GENERATION_URL="https://paste-generation-url.trycloudflare.com"
read -s "EMBEDDING_TOKEN?Embedding token: "
echo
read -s "GENERATION_TOKEN?Generation token: "
echo

curl --fail-with-body --max-time 300 -sS "$EMBEDDING_URL/embed" \
  -H "Authorization: Bearer $EMBEDDING_TOKEN" \
  -H 'Content-Type: application/json' \
  -d '{"texts":["Test from my Mac."]}'

curl --fail-with-body --max-time 300 -sS "$GENERATION_URL/generate" \
  -H "Authorization: Bearer $GENERATION_TOKEN" \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Explain retrieval augmented generation briefly.","max_new_tokens":128}'
unset EMBEDDING_TOKEN GENERATION_TOKEN
~~~

401: wrong/missing token. 413: input exceeds model limits. 422: invalid request fields. 503: missing server token or model failure; inspect the corresponding server log. Tunnel errors: check the runtime and tunnel logs. Both services share runtime resources and are not production load-tested.


## 7. Optional cleanup
Leave disabled during Run all. Set `STOP_SERVICES=True` and execute to stop this notebook's two tunnels and two servers. To restart, reset it to False, then rerun cells 3–5. Only tracked processes belonging to this notebook are stopped.


In [ ]:
STOP_SERVICES = False
if STOP_SERVICES:
    for registry_name in ("model_tunnels", "model_processes"):
        for process in globals().get(registry_name, {}).values():
            stop_process(process)
    if "MODEL_URLS" in globals():
        MODEL_URLS.clear()
    print("Both APIs and tunnels stopped.")
else:
    print("Both services stay running; cleanup is disabled.")
